# Initial-condition soil-moisture distribution

## Purpose

This notebook compares ensemble soil moisture with a gridded observational product for one target date. For each configured model experiment, it integrates soil moisture over the requested surface depth and plots:

- Top row: ensemble-mean model bias relative to the observation.
- Bottom row: ensemble standard deviation (spread).

Bias panels also report spatial pattern correlation and RMSE. Stippling marks locations where a one-sample t-test finds the ensemble-minus-observation difference significant at the configured level.

## Configuration and outputs

Routine changes belong in the User setup cell: datasets, date, depth, land masking, units, significance level, Dask chunks, contour levels, colormaps, and figure output. Required input files are checked before loading. Support files for the land mask and soil-layer thickness are reused or generated from a sample model file.


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import os
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_lnd_init"

In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)

## 1. Libraries


In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
import cmocean

## 2. Project setup


In [ ]:
from configs.initial_land_experiment_config import (
    DEFAULT_SOIL_MOISTURE_PLOT,
    get_soil_moisture_data_dict,
)
from util.initial_lnd_support import ensure_initial_land_support_files

## 3. User setup

Edit this cell for routine data and plotting changes.


In [ ]:
# =============================================================================
# USER SETUP
# Edit this cell to change datasets, date, depth, statistics, or figure style.
# =============================================================================

DATASETS = get_soil_moisture_data_dict(
    data_root=INPUT_DATA_ROOT,
    obs_root=INPUT_DATA_ROOT / "reference",
)
PLOT_CONFIG = DEFAULT_SOIL_MOISTURE_PLOT.copy()

# Input/support and output locations
LANDMASK_FILE = INPUT_DATA_ROOT / "reference/lnd_sea_mask/landmask_1x1.nc"
SOIL_LAYER_FILE = INPUT_DATA_ROOT / "reference/lnd_sea_mask/dzsoi_elm.nc"
DIAG_OUTPUT_DIR = WORKFLOW_DATA_DIR
PROCESSED_DATA_DIR = DIAG_OUTPUT_DIR / "processed"
FIGURE_DIR = WORKFLOW_FIGURE_DIR

# Variables
SOIL_MOISTURE_VARIABLE = "H2OSOI"
SOIL_LAYER_THICKNESS_VARIABLE = "DZSOI"
LAND_FRACTION_VARIABLE = "landfrac"

# Loading and computation
USE_DASK = True
DASK_CHUNKS = {"time": 1, "levgrnd": -1, "lat": 90, "lon": 180}

# Processing and figure controls
REPROCESS = False  # Rebuild the processed NetCDF cache when True.
SHOW = True
SAVE = True
BIAS_CMAP = cmocean.cm.curl
SPREAD_CMAP = cmocean.cm.deep

# Examples of common adjustments:
PLOT_CONFIG["model_keys"] = ["CTRLEN10", "CAPTEN10", "DARTEN20", "DARTEN40"]
PLOT_CONFIG["obs_key"] = "ESA_CCI"
PLOT_CONFIG["target_date"] = "2012-01-01"
PLOT_CONFIG["depth_cm"] = 5
PLOT_CONFIG["confidence_level"] = 0.05
PLOT_CONFIG["use_mass_units"] = True
PLOT_CONFIG["bias_levels"] = [-8, -6, -4, -2, -1, 0, 1, 2, 4, 6, 8]
PLOT_CONFIG["spread_levels"] = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]


## 4. Soil-moisture plotting implementation

The plotter lives in `util/initial_lnd_soil_moisture.py`.

In [ ]:
from util.initial_lnd_soil_moisture import SoilMoistureDifferencePlotter

## 5. Validation and workflow


In [ ]:
def validate_soil_moisture_inputs(data_dict, plot_config):
    target_year = pd.to_datetime(plot_config["target_date"]).year
    missing = []

    for model_key in plot_config["model_keys"]:
        info = data_dict[model_key]
        for member in range(1, info["nens"] + 1):
            filename = info["template"] % {
                "year": target_year,
                "ensemble": f"EN{member:02d}",
            }
            file_path = Path(info["path"]) / filename
            if not file_path.is_file():
                missing.append(file_path)

    obs_info = data_dict[plot_config["obs_key"]]
    obs_path = Path(obs_info["path"]) / (
        obs_info["template"] % {"year": target_year}
    )
    if not obs_path.is_file():
        missing.append(obs_path)

    if missing:
        formatted = "\n".join(f"  - {file_path}" for file_path in missing)
        raise FileNotFoundError("Missing soil-moisture inputs:\n" + formatted)


def processed_cache_path(plot_config):
    units = "mass" if plot_config["use_mass_units"] else "volumetric"
    land = "land" if plot_config["mask_land"] else "all"
    alpha = str(plot_config["confidence_level"]).replace(".", "p")
    target = plot_config["target_date"].replace("-", "")
    return PROCESSED_DATA_DIR / (
        f"sm_processed_{plot_config['obs_key']}_{target}_"
        f"top{plot_config['depth_cm']:02d}cm_{units}_{land}_alpha{alpha}.nc"
    )


def build_soil_moisture_figure():
    cache_path = processed_cache_path(PLOT_CONFIG)
    requested_models = list(PLOT_CONFIG["model_keys"])
    processed_data = None
    if cache_path.is_file() and not REPROCESS:
        candidate = xr.open_dataset(cache_path, engine="scipy").load()
        cached_models = [str(value) for value in candidate["experiment"].values]
        if cached_models == requested_models:
            processed_data = candidate
            print(f"[CACHE] Reusing processed data: {cache_path}")
        else:
            candidate.close()
            print("[CACHE] Experiment list changed; rebuilding processed data.")

    landmask_file = None
    soil_layer_file = None
    if processed_data is None:
        validate_soil_moisture_inputs(DATASETS, PLOT_CONFIG)
        landmask_file, soil_layer_file = ensure_initial_land_support_files(
            DATASETS, landmask_file=LANDMASK_FILE, soilayer_file=SOIL_LAYER_FILE,
            sample_key=PLOT_CONFIG["model_keys"][0],
            target_date=PLOT_CONFIG["target_date"],
        )

    plotter = SoilMoistureDifferencePlotter(
        data_dict=DATASETS, landmask_file=landmask_file, soilayer_file=soil_layer_file,
        variable=SOIL_MOISTURE_VARIABLE, dz_var=SOIL_LAYER_THICKNESS_VARIABLE,
        landmask_var=LAND_FRACTION_VARIABLE, depth_cm=PLOT_CONFIG["depth_cm"],
        mask_land=PLOT_CONFIG["mask_land"],
        confidence_level=PLOT_CONFIG["confidence_level"],
        use_mass_units=PLOT_CONFIG["use_mass_units"], use_dask=USE_DASK,
        dask_chunks=DASK_CHUNKS,
    )

    if processed_data is None:
        processed_data = plotter.process_bias_and_spread(
            requested_models, target_date=PLOT_CONFIG["target_date"],
            obs_key=PLOT_CONFIG["obs_key"],
        )
        PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
        processed_data.to_netcdf(
            cache_path, engine="scipy", format="NETCDF3_64BIT"
        )
        print(f"[CACHE] Saved processed data: {cache_path}")

    target = PLOT_CONFIG["target_date"].replace("-", "")
    save_path = None
    if SAVE:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = FIGURE_DIR / f"sm_bias_stddev_vs_{PLOT_CONFIG['obs_key']}_{target}.png"

    return plotter.plot_bias_and_stddev_multi_panel(
        model_keys=requested_models, target_date=PLOT_CONFIG["target_date"],
        obs_key=PLOT_CONFIG["obs_key"], bias_levels=PLOT_CONFIG["bias_levels"],
        spread_levels=PLOT_CONFIG["spread_levels"], figsize=PLOT_CONFIG["figsize"],
        fontz=PLOT_CONFIG["fontz"], cmap_bias=BIAS_CMAP, cmap_std=SPREAD_CMAP,
        savepath=save_path, show=SHOW, processed_data=processed_data,
    )


## 6. Run the diagnostic


In [ ]:
soil_moisture_figure, soil_moisture_axes = build_soil_moisture_figure()
